In [1]:
import pandas as pd
from recbole.config import Config
from UniSRec.unisrec import UniSRec
from UniSRec.data.dataset import UniSRecDataset
import numpy as np
from recbole.data.interaction import Interaction
import torch
from pathlib import Path
import pandas as pd

In [2]:
cfg_dict = {
    # paths
    "data_path": "new_folder4",
    "plm_suffix": "feat1CLS",
    "plm_size": 384,

    # training objective
    "loss_type": "CE",
    "train_neg_sample_args": None,

    # fields
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "ITEM_LIST_FIELD": "item_id_list",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"]
    },

    # sequential
    "MAX_ITEM_LIST_LENGTH": 50,
    "max_seq_length": 50,

    # model (SASRec / UniSRec)
    "hidden_size": 300,
    "inner_size": 256,
    "n_layers": 2,
    "n_heads": 2,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "layer_norm_eps": 1e-12,
    "hidden_act": "gelu",
    "initializer_range": 0.02,
    
    "train_stage": "transductive_ft",
    
    # MoE adaptor
    "n_exps": 8,
    "adaptor_layers": [384, 300],   # must match hidden_size
    "adaptor_dropout_prob": 0.2,
    "adaptor_noise": False,

    # eval
    "eval_args": {
        "split": {"RS": [0.9, 0.05, 0.05]},
        "order": "TO",
        "mode": "full"
    }
}


In [3]:
cfg_dict.update({
    # optimization
    "learning_rate": 1e-4,          # UniSRec default is relatively high
    "lr_scheduler": "cosine",
    "weight_decay": 1e-5,
    "train_batch_size": 2048,
    "benchmark_filename": None,
    "loss_type": "CE",

    # training control
    "epochs": 300,
    "eval_step": 1,                 # evaluate every epoch
    "stopping_step": 30,             # early stopping patience
    "clip_grad_norm": {
        "max_norm": 1.0,
        "norm_type": 2
    },

    # logging
    "log_wandb": False,
    "show_progress": True,

    # sequence field wiring (CRITICAL)
    "ITEM_LIST_LENGTH_FIELD": "item_length",
    "LIST_SUFFIX": "_list",
    "MAX_ITEM_LIST_LENGTH": 50,


    "train_neg_sample_args": None,
    "alias_of_item_id": None,
    "device": "cuda",

    "topk": [10,50],
    "metrics": ["Recall", "NDCG"],
    "valid_metric": "Recall@50",
    "eval_batch_size": 2048,
    "temperature": 0.07,
})

# cfg_dict["pretrained_model_path"] = "./UniSRec-FHCKM-300.pth"
config = Config(model=UniSRec, dataset="All_Beauty", config_dict=cfg_dict)


In [4]:
print("loss_type:", config["loss_type"])
print("train_neg_sample_args:", config["train_neg_sample_args"])
print("device:", config["device"])
print("pretrained_model_path:", config["pretrained_model_path"])
print("metrics:", config["metrics"])
print("topk:", config["topk"])

loss_type: CE
train_neg_sample_args: {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
device: cuda
pretrained_model_path: None
metrics: ['Recall', 'NDCG']
topk: [10, 50]


In [5]:
from recbole.config import Config
from recbole.data import data_preparation
from UniSRec.data.dataset import UniSRecDataset
from UniSRec.unisrec import UniSRec
from recbole.trainer import Trainer


dataset = UniSRecDataset(config)

# ✅ THIS creates real DataLoaders with batching
train_data, valid_data, test_data = data_preparation(config, dataset)

# sanity-check one *real* batch
batch = next(iter(train_data))
print("user_id", batch["user_id"].shape)
print("item_id_list", batch["item_id_list"].shape)
print("item_length", batch["item_length"].shape)

model = UniSRec(config, dataset).to(config["device"])

trainer = Trainer(config, model)
pretrained_loaded = any("pretrained" in k.lower() for k in trainer.saved_model_file)
print("Loaded pretrained model:", pretrained_loaded)
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.norm().item())
        break

/home/hersco/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/home/hersco/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which w

user_id torch.Size([2048])
item_id_list torch.Size([2048, 50])
item_length torch.Size([2048])
Loaded pretrained model: False
item_embedding.weight 113.65667724609375


In [6]:
print("train_data length:", len(train_data))
print("valid_data length:", len(valid_data))
print("test_data length:", len(test_data))
# print(dataset.__dict__)
print(len(dataset.field2id_token["item_id"]))

train_data length: 23
valid_data length: 2
test_data length: 4
107642


In [ ]:
# BEFORE loading - to show that the model weights are different
model_tmp = UniSRec(config, dataset).to(config["device"])
for name, p in model_tmp.named_parameters():
    if "trm_encoder.layer.0.feed_forward.dense_1.weight" in name:
        print("Random init norm:", p.norm().item())
        break


In [ ]:
import torch
import pickle

ckpt_path = "/home/hersco/RecSys/UniSRec-FHCKM-300.pth"

try:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    print("Loaded via torch.load")
except Exception as e:
    print("torch.load failed, trying pickle:", e)
    with open(ckpt_path, "rb") as f:
        ckpt = pickle.load(f)
    print("Loaded via pickle")

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("Top-level keys:", ckpt.keys())


In [ ]:
state_dict = ckpt["state_dict"]

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("Missing keys:")
for k in missing_keys:
    print("  ", k)

print("\nUnexpected keys:")
for k in unexpected_keys:
    print("  ", k)


In [ ]:
# AFTER loading - to show that the model weights have changed
print("Loaded norm:", model.trm_encoder.layer[0].feed_forward.dense_1.weight.norm().item())

In [7]:
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("\n".join(trainable[:10]))
print("Trainable params:", len(trainable))


item_embedding.weight
position_embedding.weight
trm_encoder.layer.0.multi_head_attention.query.weight
trm_encoder.layer.0.multi_head_attention.query.bias
trm_encoder.layer.0.multi_head_attention.key.weight
trm_encoder.layer.0.multi_head_attention.key.bias
trm_encoder.layer.0.multi_head_attention.value.weight
trm_encoder.layer.0.multi_head_attention.value.bias
trm_encoder.layer.0.multi_head_attention.dense.weight
trm_encoder.layer.0.multi_head_attention.dense.bias
Trainable params: 54


In [ ]:
for name, p in model.named_parameters():
    if "trm_encoder" in name:
        p.requires_grad = False


In [ ]:
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("\n".join(trainable))
print("Trainable params:", len(trainable))


In [8]:
from recbole.utils import init_logger

init_logger(config)
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, show_progress=False, verbose=True)

27 Dec 22:04    INFO  epoch 0 training [time: 6.09s, train loss: 277.1688]
27 Dec 22:04    INFO  epoch 0 evaluating [time: 0.14s, valid_score: 0.001100]
27 Dec 22:04    INFO  valid result: 
recall@10 : 0.0004    recall@50 : 0.0011    ndcg@10 : 0.0001    ndcg@50 : 0.0003
27 Dec 22:04    INFO  Saving current: saved/UniSRec-Dec-27-2025_22-04-07.pth
27 Dec 22:04    INFO  epoch 1 training [time: 5.72s, train loss: 265.4880]
27 Dec 22:04    INFO  epoch 1 evaluating [time: 0.12s, valid_score: 0.002300]
27 Dec 22:04    INFO  valid result: 
recall@10 : 0.0011    recall@50 : 0.0023    ndcg@10 : 0.0005    ndcg@50 : 0.0008
27 Dec 22:04    INFO  Saving current: saved/UniSRec-Dec-27-2025_22-04-07.pth
27 Dec 22:04    INFO  epoch 2 training [time: 5.74s, train loss: 262.0020]
27 Dec 22:04    INFO  epoch 2 evaluating [time: 0.11s, valid_score: 0.008300]
27 Dec 22:04    INFO  valid result: 
recall@10 : 0.0023    recall@50 : 0.0083    ndcg@10 : 0.0012    ndcg@50 : 0.0025
27 Dec 22:04    INFO  Saving curr

In [9]:
print("===== TRAIN LOSS PER EPOCH =====")
for epoch, loss in trainer.train_loss_dict.items():
    print(f"Epoch {epoch}: loss = {loss:.6f}")

===== TRAIN LOSS PER EPOCH =====
Epoch 0: loss = 277.168848
Epoch 1: loss = 265.488013
Epoch 2: loss = 262.002037
Epoch 3: loss = 256.136838
Epoch 4: loss = 246.474272
Epoch 5: loss = 238.688623
Epoch 6: loss = 233.952286
Epoch 7: loss = 231.592262
Epoch 8: loss = 230.107158
Epoch 9: loss = 229.737360
Epoch 10: loss = 228.741735
Epoch 11: loss = 228.363246
Epoch 12: loss = 227.982416
Epoch 13: loss = 227.413418
Epoch 14: loss = 227.144310
Epoch 15: loss = 226.907914
Epoch 16: loss = 226.759543
Epoch 17: loss = 226.484931
Epoch 18: loss = 226.246971
Epoch 19: loss = 226.085659
Epoch 20: loss = 225.649855
Epoch 21: loss = 225.338692
Epoch 22: loss = 224.823229
Epoch 23: loss = 224.494454
Epoch 24: loss = 223.981701
Epoch 25: loss = 223.154280
Epoch 26: loss = 221.926981
Epoch 27: loss = 220.781018
Epoch 28: loss = 219.831482
Epoch 29: loss = 219.046021
Epoch 30: loss = 217.856757
Epoch 31: loss = 216.465919
Epoch 32: loss = 214.402665
Epoch 33: loss = 212.547325
Epoch 34: loss = 210.7077

In [10]:
print("\n===== BEST VALIDATION =====")
print("Best valid score:", best_valid_score)
for k, v in best_valid_result.items():
    print(f"{k}: {v}")



===== BEST VALIDATION =====
Best valid score: 0.0791
recall@10: 0.0314
recall@50: 0.0791
ndcg@10: 0.0152
ndcg@50: 0.0254


In [8]:
import torch

ckpt_path = "saved/UniSRec-Dec-27-2025_22-04-07.pth"
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
model.to(config["device"])
model.eval()


UniSRec(
  (item_embedding): Embedding(107642, 300, padding_idx=0)
  (position_embedding): Embedding(50, 300)
  (trm_encoder): TransformerEncoder(
    (layer): ModuleList(
      (0-1): 2 x TransformerLayer(
        (multi_head_attention): MultiHeadAttention(
          (query): Linear(in_features=300, out_features=300, bias=True)
          (key): Linear(in_features=300, out_features=300, bias=True)
          (value): Linear(in_features=300, out_features=300, bias=True)
          (softmax): Softmax(dim=-1)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (dense): Linear(in_features=300, out_features=300, bias=True)
          (LayerNorm): LayerNorm((300,), eps=1e-12, elementwise_affine=True)
          (out_dropout): Dropout(p=0.2, inplace=False)
        )
        (feed_forward): FeedForward(
          (dense_1): Linear(in_features=300, out_features=256, bias=True)
          (dense_2): Linear(in_features=256, out_features=300, bias=True)
          (LayerNorm): LayerNorm((3

In [9]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from recbole.data.interaction import Interaction

# ------------------------------
# 0) Make CUDA errors synchronous (so stacktraces point to the real line)
# ------------------------------
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ------------------------------
# 1) Load model weights safely
# ------------------------------
DEVICE = config["device"]
ckpt_path = "saved/UniSRec-Dec-27-2025_22-04-07.pth"
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
model.to(DEVICE)
model.eval()

# ------------------------------
# 2) Fields & mappings (authoritative)
# ------------------------------
uid_field = dataset.uid_field
iid_field = dataset.iid_field
item_seq_field = model.ITEM_SEQ
item_len_field = model.ITEM_SEQ_LEN

token2id = dataset.field2token_id[iid_field]   # ASIN -> internal id
id2token = dataset.field2id_token[iid_field]   # internal id -> ASIN

PAD_ID = 0
MAX_LEN = config["MAX_ITEM_LIST_LENGTH"]
TOP_K = 10
BATCH = 256

# sanity: vocab/emb sizes match
n_items = len(id2token)
assert n_items == model.item_embedding.num_embeddings == model.plm_embedding.num_embeddings, \
    (n_items, model.item_embedding.num_embeddings, model.plm_embedding.num_embeddings)

# ------------------------------
# 3) Load test
# ------------------------------
test_df = pd.read_csv("All_Beauty.test.csv")
assert "id" in test_df.columns and "history" in test_df.columns
N = len(test_df)

# ------------------------------
# 4) Build sequences (PAD-left, right-aligned) + CRITICAL: len>=1
# ------------------------------
seqs = np.full((N, MAX_LEN), PAD_ID, dtype=np.int64)
lens = np.zeros(N, dtype=np.int64)

empty_hist_count = 0
unk_tok_count = 0

for i, row in test_df.iterrows():
    hist = str(row["history"]) if pd.notna(row["history"]) else ""
    toks = hist.split()

    ids = []
    for a in toks:
        if a in token2id:
            ids.append(int(token2id[a]))
        else:
            unk_tok_count += 1

    ids = ids[-MAX_LEN:]

    # 🔴 CRITICAL FIX: enforce length >= 1
    if len(ids) == 0:
        empty_hist_count += 1
        ids = [PAD_ID]

    lens[i] = len(ids)
    seqs[i, -len(ids):] = ids  # right-align

print(f"Empty histories fixed: {empty_hist_count} / {N}")
print(f"Unknown tokens skipped (not in vocab): {unk_tok_count}")

# hard bounds checks (must pass)
assert lens.min() >= 1
assert lens.max() <= MAX_LEN
assert seqs.min() >= 0
assert seqs.max() < n_items

# dummy user ids (some models require it)
uids = np.zeros(N, dtype=np.int64)

# ------------------------------
# 5) Batched inference + robust filtering
# ------------------------------
all_preds = []

with torch.no_grad():
    for start in tqdm(range(0, N, BATCH), desc="Inference"):
        end = min(N, start + BATCH)

        inter = Interaction({
            uid_field: torch.from_numpy(uids[start:end]).long(),
            item_seq_field: torch.from_numpy(seqs[start:end]).long(),
            item_len_field: torch.from_numpy(lens[start:end]).long(),
        }).to(DEVICE)

        scores = model.full_sort_predict(inter)  # (B, n_items)

        top_ids = torch.topk(scores, k=TOP_K + 100, dim=1).indices.cpu().numpy()

        for r in range(end - start):
            seen = set(seqs[start + r].tolist())
            recs = []
            for idx in top_ids[r]:
                idx = int(idx)
                if idx != PAD_ID and idx not in seen:
                    recs.append(id2token[idx])  # internal id -> ASIN
                if len(recs) == TOP_K:
                    break

            # fallback (should be rare)
            if len(recs) < TOP_K:
                for idx in top_ids[r]:
                    idx = int(idx)
                    if idx != PAD_ID and id2token[idx] not in recs:
                        recs.append(id2token[idx])
                    if len(recs) == TOP_K:
                        break

            assert len(recs) == TOP_K
            all_preds.append(recs)

# ------------------------------
# 6) Build submission
# ------------------------------
submission = pd.DataFrame({"id": test_df["id"].values})
for i in range(TOP_K):
    submission[f"rec{i+1}"] = [row[i] for row in all_preds]

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv ready")
print(submission.head())


Empty histories fixed: 824 / 4778
Unknown tokens skipped (not in vocab): 2571


Inference: 100%|██████████| 19/19 [00:01<00:00, 16.66it/s]


✅ submission.csv ready
   id        rec1        rec2        rec3        rec4        rec5        rec6  \
0   0  B08YJ65SQX  B08532CNY7  B08342YK4F  B098DR3D8V  B07V5DV9VZ  B08Q7D4B8M   
1   1  B08M3Y94KW  B07VFF1B25  B083B67373  B08DXPNCV1  B07B51187X  B08BFGHMW7   
2   2  B08M3Y94KW  B07VFF1B25  B083B67373  B08DXPNCV1  B07B51187X  B08BFGHMW7   
3   3  B08M3Y94KW  B07VFF1B25  B083B67373  B08DXPNCV1  B07B51187X  B08BFGHMW7   
4   4  B08SBV6Z57  B08XYLQTX2  B07DPMKDVH  B08LDBM631  B08MXT57Z8  B084GF7BSV   

         rec7        rec8        rec9       rec10  
0  B08G2MDZ3X  B081L8J1D8  B000E8P2OW  B01DU97Q8Q  
1  B08JP74M51  B08HGZXLP6  B08MTW68VR  B089NJG212  
2  B08JP74M51  B08HGZXLP6  B08MTW68VR  B089NJG212  
3  B08JP74M51  B08HGZXLP6  B08MTW68VR  B089NJG212  
4  B093P3GKF6  B08T1TTXV2  B08MF63Q6F  B08HLGW85W  


In [8]:
n_dataset = len(dataset.field2id_token["item_id"])
n_item_emb = model.item_embedding.num_embeddings
n_plm_emb = model.plm_embedding.num_embeddings

print("dataset items:", n_dataset)
print("item embedding:", n_item_emb)
print("plm embedding:", n_plm_emb)

assert n_dataset <= n_item_emb
assert n_dataset <= n_plm_emb


dataset items: 107642
item embedding: 107642
plm embedding: 107642


In [ ]:
import pandas as pd

test_df = pd.read_csv("All_Beauty.test.csv")

print("Test CSV rows:", len(test_df))
print("Unique ids:", test_df["id"].nunique())
print("Min id:", test_df["id"].min(), "Max id:", test_df["id"].max())


In [ ]:
import numpy as np
model.eval()
device = config["device"]
PAD_ID = 0
K = 10
MAX_LEN = config["MAX_ITEM_LIST_LENGTH"]

# Load Kaggle test
test_df = pd.read_csv("All_Beauty.test.csv")

id2token = dataset.field2id_token["item_id"]
token2id = {tok: i for i, tok in enumerate(id2token)}
n_items = model.n_items

# Build sequences (CPU)
seqs = np.zeros((len(test_df), MAX_LEN), dtype=np.int64)
lens = np.zeros(len(test_df), dtype=np.int64)

for i, row in test_df.iterrows():
    hist = str(row["history"]) if pd.notna(row["history"]) else ""
    ids = [token2id[a] for a in hist.split() if a in token2id]
    ids = ids[-MAX_LEN:]
    lens[i] = len(ids)
    if ids:
        seqs[i, -len(ids):] = ids

# Sanity check (this prevents CUDA assert)
assert seqs.min() >= 0
assert seqs.max() < n_items


In [ ]:
# config = Config(model=UniSRec, dataset="All_Beauty", config_dict=cfg_dict)
# dataset = UniSRecDataset(config)
# model = UniSRec(config, dataset).to(config["device"])
# model.eval()

device = config["device"]
PAD_ID = 0
K = 10
MAX_LEN = config["MAX_ITEM_LIST_LENGTH"]

# ------------ mappings (authoritative) ------------
# id2token = dataset.field2id_token["item_id"]            # id -> asin
# token2id_map = dataset.field2token_id["item_id"]        # asin -> id (authoritative)

# Read from ./new_folder2/All_Beauty/All_Beauty.item2index
id2token = []
token2id_map = {}
with open("new_folder2/All_Beauty/All_Beauty.item2index", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) != 2:
            continue
        asin, idx = parts
        idx = int(idx)
        while len(id2token) <= idx:
            id2token.append("[PAD]")
        id2token[idx] = asin
        token2id_map[asin] = idx


def tok2id(tok: str) -> int:
    try:
        return int(token2id_map[tok])
    except Exception:
        print("⚠️  Unknown token:", tok)
        return 0

def id2tok(idx: int) -> str:
    try:
        return str(id2token[idx])
    except Exception:
        return "[PAD]"

# embedding table sizes (REAL truth)
n_plm = model.plm_embedding.num_embeddings
n_item = model.item_embedding.num_embeddings
print("plm_embedding.num_embeddings:", n_plm)
print("item_embedding.num_embeddings:", n_item)
print("len(id2token):", len(id2token))

# ------------ load test ------------
test_df = pd.read_csv("All_Beauty.test.csv")
assert "id" in test_df.columns and "history" in test_df.columns
N = len(test_df)
print("Test rows:", N)

# ------------ build sequences on CPU ------------
seqs = np.zeros((N, MAX_LEN), dtype=np.int64)
lens = np.zeros(N, dtype=np.int64)

for i, row in test_df.iterrows():
    hist = str(row["history"]) if pd.notna(row["history"]) else ""
    ids = [tok2id(a) for a in hist.split()]
    # clamp to PLM embedding range (this prevents CUDA asserts)
    ids = [x if (0 <= x < n_plm) else 0 for x in ids]
    ids = ids[-MAX_LEN:]
    lens[i] = len(ids)
    if ids:
        seqs[i, -len(ids):] = ids

# hard sanity
print("seqs min/max:", int(seqs.min()), int(seqs.max()))
assert seqs.min() >= 0
assert seqs.max() < n_plm
assert seqs.max() < n_item

# ------------ batched GPU inference (avoid huge N x items blowups) ------------
BATCH = 256
all_pred_asins = []

with torch.no_grad():
    for start in range(0, N, BATCH):
        end = min(N, start + BATCH)
        seqs_b = torch.from_numpy(seqs[start:end]).long().to(device)
        lens_b = torch.from_numpy(lens[start:end]).long().to(device)

        inter = Interaction({"item_id_list": seqs_b, "item_length": lens_b})

        scores = model.full_sort_predict(inter)  # (B, n_items)
        if np.random.rand() < 0.01:
            print("Scores sample:", scores[0, :10])
        # move to CPU for safe filtering
        topk_ids = torch.topk(scores, k=200, dim=1).indices.cpu().numpy()  # take more, filter later

        # CPU filtering: remove PAD and remove items already in history
        for r in range(end - start):
            seen = set(seqs[start + r].tolist())
            row = []
            for x in topk_ids[r]:
                x = int(x)
                if x != PAD_ID and x not in seen:
                    row.append(x)
                if len(row) == K:
                    break
            # fallback if somehow not enough (should be rare)
            if len(row) < K:
                print("⚠️  Not enough recommendations, applying fallback")
                for x in topk_ids[r]:
                    x = int(x)
                    if x != PAD_ID and x not in row:
                        row.append(x)
                    if len(row) == K:
                        break

            all_pred_asins.append([id2tok(x) for x in row])

# final shape checks
assert len(all_pred_asins) == N
assert all(len(r) == K for r in all_pred_asins)
assert all("[PAD]" not in r for r in all_pred_asins)

print("Sample predictions:")
for i in range(5):
    print(all_pred_asins[i])

# ------------ build Kaggle submission ------------
submission = pd.DataFrame({"id": test_df["id"].values})
for i in range(K):
    submission[f"rec{i+1}"] = [row[i] for row in all_pred_asins]

assert len(submission) == len(test_df)
assert submission["id"].equals(test_df["id"])

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv ready:", submission.shape)
print(submission.head())


In [ ]:
dataset.__dict__

In [ ]:
import pandas as pd

submission = pd.read_csv("submission.csv")

for k in range(1, 11):
    col = f"rec{k}"
    submission[col] = submission[col].apply(
        lambda x: id2asin.get(int(x), "[UNK]")
    )

assert not submission.isin(["[UNK]"]).any().any(), "❌ Unknown item IDs found"

submission.to_csv("submission.csv", index=False)
print("✅ ASIN conversion done")


In [ ]:
import json
from pathlib import Path

jsonl_path = Path("meta_All_Beauty.jsonl")
out_path = Path("new_folder/All_Beauty/item_text.json")

item_text = {}

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        asin = obj.get("parent_asin") or obj.get("asin")
        if asin is None:
            continue
        item_text[asin] = obj

out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(item_text, f)

print("✅ item_text.json written")
print("Items:", len(item_text))
